# Семинар 5. Валидация и подбор гиперпараметров

В этом семинаре мы разберем:
- Правильное разделение данных (train/validation/test)
- Кросс-валидация (K-fold, Stratified K-fold, LOO, Time Series)
- Learning curves и validation curves
- Grid Search, Random Search, Optuna
- Сравнение методов на тестовой выборке

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q catboost optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    KFold, StratifiedKFold, LeaveOneOut, TimeSeriesSplit,
    cross_val_score, learning_curve, validation_curve,
)
from sklearn.metrics import accuracy_score, classification_report
from catboost import CatBoostClassifier
import optuna

np.random.seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Загрузка и подготовка данных

Используем [Student Depression Dataset](https://www.kaggle.com/datasets/adilshamim8/student-depression-dataset) - 27 901 студент, 17 признаков, бинарный таргет (депрессия).

In [ ]:
df = pd.read_csv('student_depression.csv')
print("Размер датасета:", df.shape)

# Удаляем id, кодируем строковые столбцы числами для совместимости с sklearn CV
df = df.drop('id', axis=1)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Категориальные признаки ({len(cat_cols)}): {cat_cols}")

from sklearn.preprocessing import LabelEncoder
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

df.head()

## 2. Разделение данных

При подборе гиперпараметров важно правильно разделить данные на три части:
- **Train** - для обучения модели
- **Validation** - для подбора гиперпараметров
- **Test** - для финальной оценки модели

Распространенная ошибка: подбирать гиперпараметры на тесте. Это приводит к оптимистичной оценке - модель "подстроилась" под тест через гиперпараметры.

In [ ]:
X = df.drop('Depression', axis=1)
y = df['Depression']

# Сначала отделяем тестовую выборку (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Затем разделяем оставшиеся данные на train (60%) и validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")
print(f"\nРаспределение классов в train: {y_train.value_counts(normalize=True).to_dict()}")

## 3. Кросс-валидация

Вместо одного фиксированного разбиения на train/val можно использовать кросс-валидацию: данные разбиваются на K частей, модель обучается K раз, каждый раз валидация на новой части.

### 3.1 K-fold Cross-validation

Данные разбиваются на K равных частей. Модель обучается K раз: каждый раз K-1 частей для обучения, 1 для валидации.

In [ ]:
def plot_cv_splits(X, cv, title):
    """Visualize cross-validation splits."""
    n_samples = len(X)
    splits = list(cv.split(X, y_temp) if hasattr(cv, 'split') else cv.split(X))
    n_splits = len(splits)
    fig, axs = plt.subplots(n_splits, 1, figsize=(15, 1.5 * n_splits))
    fig.suptitle(title, fontsize=14)
    for idx, (train_idx, val_idx) in enumerate(splits):
        axs[idx].set_yticks([])
        axs[idx].set_xlim(0, n_samples)
        mask = np.zeros(n_samples)
        mask[val_idx] = 1
        axs[idx].imshow([mask], aspect='auto', cmap='coolwarm', vmin=0, vmax=1,
                        extent=[0, n_samples, -0.5, 0.5])
        axs[idx].set_ylabel(f'Fold {idx+1}', rotation=0, labelpad=40)
    axs[-1].set_xlabel('Sample index (blue=train, red=val)')
    plt.tight_layout()
    plt.show()

In [ ]:
base_model = CatBoostClassifier(iterations=100, verbose=False, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
plot_cv_splits(X_temp, kf, 'K-fold Cross-validation (K=5)')

kf_scores = cross_val_score(base_model, X_temp, y_temp, cv=kf, scoring='accuracy')
print(f"K-fold scores: {kf_scores}")
print(f"Mean: {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}")

### 3.2 Stratified K-fold

Сохраняет распределение классов в каждом фолде. Важно при несбалансированных данных.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
plot_cv_splits(X_temp, skf, 'Stratified K-fold Cross-validation (K=5)')

skf_scores = cross_val_score(base_model, X_temp, y_temp, cv=skf, scoring='accuracy')
print(f"Stratified K-fold scores: {skf_scores}")
print(f"Mean: {skf_scores.mean():.4f} +/- {skf_scores.std():.4f}")

# Проверяем распределение классов в фолдах
print(f"\nОбщее распределение: {y_temp.value_counts(normalize=True).to_dict()}")
for i, (train_idx, val_idx) in enumerate(skf.split(X_temp, y_temp)):
    fold_dist = y_temp.iloc[val_idx].value_counts(normalize=True).to_dict()
    print(f"Fold {i+1} val: {fold_dist}")

### 3.3 Leave-One-Out (LOO)

Предельный случай K-fold с K = N. На каждой итерации 1 объект для валидации, остальные для обучения. Очень дорого вычислительно - демонстрируем на подвыборке.

In [ ]:
small_X = X_temp.iloc[:50]
small_y = y_temp.iloc[:50]

loo = LeaveOneOut()
loo_scores = []

for train_idx, val_idx in loo.split(small_X):
    model = CatBoostClassifier(iterations=50, verbose=False, random_state=42)
    model.fit(small_X.iloc[train_idx], small_y.iloc[train_idx])
    pred = model.predict(small_X.iloc[val_idx])
    loo_scores.append(accuracy_score(small_y.iloc[val_idx], pred))

print(f"LOO CV mean accuracy: {np.mean(loo_scores):.4f}")
print(f"LOO потребовал {len(loo_scores)} обучений модели")

### 3.4 Time Series Split

Для временных рядов обычный K-fold нарушает временной порядок (будущее попадает в train). TimeSeriesSplit всегда обучается на прошлом, валидируется на будущем.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
plot_cv_splits(X_temp, tscv, 'Time Series Cross-validation')

ts_scores = cross_val_score(base_model, X_temp, y_temp, cv=tscv, scoring='accuracy')
print(f"Time Series CV scores: {ts_scores}")
print(f"Mean: {ts_scores.mean():.4f} +/- {ts_scores.std():.4f}")

### Когда какой метод использовать

| Метод | Когда использовать |
|---|---|
| K-fold | По умолчанию, достаточно данных |
| Stratified K-fold | Несбалансированные классы |
| LOO | Очень мало данных (<100) |
| Time Series Split | Данные с временной структурой |

## 4. Learning curve и Validation curve

Два диагностических инструмента для понимания поведения модели.

### 4.1 Learning curve

Показывает, как меняется качество с ростом объема обучающей выборки. Помогает ответить на вопрос: "поможет ли больше данных?"

- Если train и test scores сходятся к высокому значению - данных достаточно
- Если между ними большой разрыв - модель переобучается, нужно больше данных или упрощение модели
- Если оба низкие - модель недообучается, нужна более сложная модель

In [ ]:
model_lc = CatBoostClassifier(iterations=100, depth=6, verbose=False, random_state=42)

train_sizes, train_scores_lc, val_scores_lc = learning_curve(
    model_lc, X_temp, y_temp,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=3, scoring='accuracy', n_jobs=-1,
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores_lc.mean(axis=1), 'o-', label='Train')
plt.plot(train_sizes, val_scores_lc.mean(axis=1), 'o-', label='Validation')
plt.fill_between(train_sizes,
                 train_scores_lc.mean(axis=1) - train_scores_lc.std(axis=1),
                 train_scores_lc.mean(axis=1) + train_scores_lc.std(axis=1), alpha=0.1)
plt.fill_between(train_sizes,
                 val_scores_lc.mean(axis=1) - val_scores_lc.std(axis=1),
                 val_scores_lc.mean(axis=1) + val_scores_lc.std(axis=1), alpha=0.1)
plt.xlabel('Training set size')
plt.ylabel('Accuracy')
plt.title('Learning Curve (CatBoost, depth=6)')
plt.legend()
plt.grid(True)
plt.show()

### 4.2 Validation curve

Показывает, как меняется качество при изменении одного гиперпараметра. Помогает увидеть зону underfitting/overfitting.

In [ ]:
depths = np.arange(1, 12)

train_scores_vc, val_scores_vc = validation_curve(
    CatBoostClassifier(iterations=100, verbose=False, random_state=42),
    X_temp, y_temp,
    param_name='depth', param_range=depths,
    cv=3, scoring='accuracy', n_jobs=-1,
)

plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores_vc.mean(axis=1), 'o-', label='Train')
plt.plot(depths, val_scores_vc.mean(axis=1), 'o-', label='Validation')
plt.fill_between(depths,
                 train_scores_vc.mean(axis=1) - train_scores_vc.std(axis=1),
                 train_scores_vc.mean(axis=1) + train_scores_vc.std(axis=1), alpha=0.1)
plt.fill_between(depths,
                 val_scores_vc.mean(axis=1) - val_scores_vc.std(axis=1),
                 val_scores_vc.mean(axis=1) + val_scores_vc.std(axis=1), alpha=0.1)
plt.xlabel('Tree depth')
plt.ylabel('Accuracy')
plt.title('Validation Curve: depth')
plt.legend()
plt.grid(True)
plt.show()

## 5. Grid Search

Перебирает все комбинации заданных значений гиперпараметров.

- [+] Гарантированно находит лучшую комбинацию из заданных
- [-] Вычислительно затратный (экспоненциальный рост)
- [-] Может пропустить оптимум между точками сетки

In [ ]:
param_grid = {
    'learning_rate': [0.01, 0.1],
    'depth': [4, 6, 8],
    'iterations': [100, 200],
}

grid_search = GridSearchCV(
    CatBoostClassifier(verbose=False, random_state=42),
    param_grid=param_grid,
    cv=3, n_jobs=-1, scoring='accuracy',
)
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший score (CV): {grid_search.best_score_:.4f}")
print(f"Проверено комбинаций: {len(grid_search.cv_results_['params'])}")

## 6. Random Search

Случайно выбирает комбинации из заданных распределений.

- [+] Эффективнее Grid Search при том же бюджете итераций
- [+] Может найти неожиданно хорошие комбинации
- [-] Не гарантирует нахождение оптимума

In [ ]:
param_distributions = {
    'learning_rate': np.logspace(-3, 0, 1000),
    'depth': range(3, 10),
    'iterations': range(100, 500),
}

random_search = RandomizedSearchCV(
    CatBoostClassifier(verbose=False, random_state=42),
    param_distributions=param_distributions,
    n_iter=20, cv=3, n_jobs=-1,
    scoring='accuracy', random_state=42,
)
random_search.fit(X_train, y_train)

print(f"Лучшие параметры: {random_search.best_params_}")
print(f"Лучший score (CV): {random_search.best_score_:.4f}")

## 7. Optuna (байесовская оптимизация)

Использует историю предыдущих экспериментов для выбора следующих точек. Более эффективен, чем Grid/Random Search.

- [+] Учитывает результаты предыдущих итераций
- [+] Поддерживает раннюю остановку неперспективных экспериментов (pruning)
- [+] Может оптимизировать непрерывные, дискретные и категориальные параметры

In [ ]:
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 9),
        'iterations': trial.suggest_int('iterations', 100, 500),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
    }
    model = CatBoostClassifier(**params, verbose=False, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Лучшие параметры: {study.best_params}")
print(f"Лучший score (CV): {study.best_value:.4f}")

In [ ]:
# Визуализация процесса оптимизации
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# История значений
trials = study.trials
values = [t.value for t in trials]
axes[0].plot(values, 'o-', alpha=0.7)
axes[0].axhline(y=study.best_value, color='r', linestyle='--', label=f'best={study.best_value:.4f}')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Optuna: optimization history')
axes[0].legend()
axes[0].grid(True)

# Важность гиперпараметров
importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()))
axes[1].set_xlabel('Importance')
axes[1].set_title('Optuna: hyperparameter importance')
axes[1].grid(True, axis='x')

plt.tight_layout()
plt.show()

## 8. Сравнение на тестовой выборке

Финальная оценка - только на тесте, который ни разу не использовался при обучении или подборе гиперпараметров.

In [ ]:
def evaluate_model(params, name):
    model = CatBoostClassifier(**params, verbose=False, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = accuracy_score(y_test, y_pred)
    print(f"{name}: test accuracy = {score:.4f}")
    return score

print("--- Финальное сравнение на тестовой выборке ---\n")
scores = {}
scores['Baseline'] = evaluate_model({'iterations': 100}, 'Baseline (default)')
scores['Grid Search'] = evaluate_model(grid_search.best_params_, 'Grid Search')
scores['Random Search'] = evaluate_model(random_search.best_params_, 'Random Search')
scores['Optuna'] = evaluate_model(study.best_params, 'Optuna')

plt.figure(figsize=(10, 5))
plt.bar(scores.keys(), scores.values())
plt.ylabel('Test Accuracy')
plt.title('Comparison of hyperparameter tuning methods')
plt.ylim(min(scores.values()) - 0.01, max(scores.values()) + 0.01)
plt.grid(True, axis='y')
plt.show()

## Выводы

1. **Разделение данных** - всегда держите тестовую выборку отдельно. Подбор гиперпараметров - на валидации или через кросс-валидацию.
2. **Кросс-валидация** дает более надежную оценку, чем одно разбиение. Stratified K-fold - дефолтный выбор для классификации.
3. **Learning curve** отвечает на вопрос "поможет ли больше данных?" Validation curve - "в какую сторону крутить гиперпараметр?"
4. **Grid Search** - простой, но дорогой. Подходит для 2-3 параметров с несколькими значениями.
5. **Random Search** - при том же бюджете находит решения не хуже Grid Search (часто лучше).
6. **Optuna** - умнее случайного поиска, учитывает историю. Лучший выбор при большом пространстве параметров.